In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.kolozov.lesson3 import Exercise


def normalize_data(X_train, X_val, X_test):
    X_train_norm = X_train.astype(np.float32) / 16.0
    X_val_norm = X_val.astype(np.float32) / 16.0
    X_test_norm = X_test.astype(np.float32) / 16.0
    return X_train_norm, X_val_norm, X_test_norm


def evaluate_model(
    lr: float,
    batch_size: int | None,
    architecture: list,
    X_train: np.ndarray,
    X_val: np.ndarray,
    y_train: np.ndarray,
    y_val: np.ndarray,
    n_epoch: int,
    seed: int,
) -> float:
    rng = np.random.default_rng(seed)

    layers = []
    for layer_config in architecture:
        layer_type = layer_config["type"]
        if layer_type == "linear":
            layers.append(Exercise.create_linear_layer(int(layer_config["in"]), int(layer_config["out"]), rng=rng))
        elif layer_type == "relu":
            layers.append(Exercise.create_relu_layer())
        elif layer_type == "sigmoid":
            layers.append(Exercise.create_sigmoid_layer())
        elif layer_type == "logsoftmax":
            layers.append(Exercise.create_logsoftmax_layer())

    model = Exercise.create_model(*layers)
    loss_fn = Exercise.create_cross_entropy_loss()

    Exercise.train_model(
        model=model,
        loss=loss_fn,
        x=X_train,
        y=y_train,
        lr=lr,
        n_epoch=n_epoch,
        batch_size=batch_size if batch_size is not None else len(X_train),
    )

    acc_val = compute_accuracy(model, X_val, y_val)
    return acc_val


def compute_accuracy(model, X: np.ndarray, y: np.ndarray) -> float:
    logits = model.forward(X.astype(np.float32))
    y_pred = np.argmax(logits, axis=1)
    return np.mean(y_pred == y)


def main():
    digits = load_digits()
    X = digits.data
    y = digits.target

    print(f"Всего образцов: {len(X)}")

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}\n")

    X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

    architecture = [
        {"type": "linear", "in": 64, "out": 256},
        {"type": "relu"},
        {"type": "linear", "in": 256, "out": 10},
    ]

    n_epoch = 10
    seeds = [42, 123, 456]

    lr_values = [0.003, 0.01, 0.03, 0.1]
    batch_size_values = [2, 4, 8, 16, 32, 64, None]

    best_config = {
        "lr": 0,
        "batch_size": None,
        "acc_mean": 0.0,
        "acc_std": 1.0,
    }
    results = []

    for lr in lr_values:
        for batch_size in batch_size_values:
            acc_scores = []

            for seed in seeds:
                acc = evaluate_model(
                    lr=lr,
                    batch_size=batch_size,
                    architecture=architecture,
                    X_train=X_train_norm,
                    X_val=X_val_norm,
                    y_train=y_train,
                    y_val=y_val,
                    n_epoch=n_epoch,
                    seed=seed,
                )
                acc_scores.append(acc)

            acc_mean = np.mean(acc_scores)
            acc_std = np.std(acc_scores)

            results.append((lr, batch_size, acc_mean, acc_std))

            if acc_mean > best_config["acc_mean"] or (
                acc_mean == best_config["acc_mean"] and acc_std < best_config["acc_std"]
            ):
                best_config.update(
                    {
                        "lr": lr,
                        "batch_size": batch_size,
                        "acc_mean": acc_mean,
                        "acc_std": acc_std,
                    }
                )

    results.sort(key=lambda x: (-x[2], x[3]))

    print("\n" + "=" * 80)
    print(f"{'ТОП КОМБИНАЦИЙ ПО ВАЛИДАЦИИ':^80}")
    print("=" * 80)
    print(f"{'№':<3} {'lr':<8} {'batch_size':<10} {'Acc mean':<10} {'Acc std':<10}")
    print("-" * 80)

    for i, (lr, bs, mean, std) in enumerate(results[:10], 1):
        bs_str = str(bs) if bs is not None else "full"
        print(f"{i:<3} {lr:<8} {bs_str:<10} {mean:.4f}       ±{std:.4f}")

    print("=" * 80)

    print("\n" + "=" * 80)
    print("ЛУЧШАЯ КОНФИГУРАЦИЯ")
    print("=" * 80)
    print(f"Learning Rate:      {best_config['lr']}")
    print(f"Batch Size:         {best_config['batch_size'] if best_config['batch_size'] else 'full batch'}")
    print(f"Val Accuracy:       {best_config['acc_mean']:.4f} ± {best_config['acc_std']:.4f}")
    print(f"Эпох:               {n_epoch}")

    print("\n" + "=" * 80)
    print("ФИНАЛЬНАЯ ПРОВЕРКА НА ТЕСТЕ")
    print("=" * 80)

    final_rng = np.random.default_rng(42)

    layers = []
    for layer_config in architecture:
        layer_type = layer_config["type"]
        if layer_type == "linear":
            layers.append(
                Exercise.create_linear_layer(int(layer_config["in"]), int(layer_config["out"]), rng=final_rng)
            )
        elif layer_type == "relu":
            layers.append(Exercise.create_relu_layer())
        elif layer_type == "sigmoid":
            layers.append(Exercise.create_sigmoid_layer())
        elif layer_type == "logsoftmax":
            layers.append(Exercise.create_logsoftmax_layer())

    final_model = Exercise.create_model(*layers)
    final_loss = Exercise.create_cross_entropy_loss()

    Exercise.train_model(
        model=final_model,
        loss=final_loss,
        x=X_train_norm,
        y=y_train,
        lr=best_config["lr"],
        n_epoch=n_epoch,
        batch_size=best_config["batch_size"] if best_config["batch_size"] else len(X_train_norm),
    )

    test_acc = compute_accuracy(final_model, X_test_norm, y_test)
    val_acc = best_config["acc_mean"]

    print(f"Test Accuracy:      {test_acc:.4f}")
    print(f"Val Accuracy:       {val_acc:.4f}")
    print(f"Gap (Val - Test):   {val_acc - test_acc:+.4f}")

    print("\n" + "=" * 80)

    return best_config


if __name__ == "__main__":
    main()